# Next generation sequencing data and population dynamics for novel GNRA/receptors isolated by in vitro selection and evolution Exploration with `mlcroissant`
This notebook demonstrates stepwise exploration and processing of the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.sgrj-01tk/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.sgrj-01tk/fair2.json"

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Version: {metadata.version}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List available record sets and their fields, referenced by `@id`

# Fetch the list of Record Set @ids
record_sets = [rs['@id'] for rs in metadata.recordSet]

print("RecordSet IDs:")
for rec_id in record_sets:
    print(f"  - {rec_id}")
    record_set_obj = next(rs for rs in metadata.recordSet if rs['@id']==rec_id)
    fields = [field['@id'] for field in record_set_obj['field']]
    print("    Fields:")
    for f in fields:
        print(f"      - {f}")

# Show sample records from the first Record Set
first_record_set_id = record_sets[0] if record_sets else None
if first_record_set_id:
    print(f"\nSample records from record set '{first_record_set_id}':")
    for x in dataset.records(record_set=first_record_set_id):
        print(x)
        break  # Show only one sample record

## 3. Data Extraction
Load data from each record set into a DataFrame. Use the record set and field `@id`s from the overview. All column references must be by their `@id`.

In [ ]:
# Extract data from all record sets
dataframes = {}
for rec_id in record_sets:
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df

# Display columns for the first record set
if first_record_set_id in dataframes:
    print(f"Columns (fields by @id) for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    print("\nFirst 5 records:")
    print(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filtering, normalizing numeric columns, grouping by field `@id`, and removing outliers.

Choose a numeric field for analysis. All field/column references will use the `@id`.

In [ ]:
from numpy import nan

# Choose a record set and its numeric field by @id.

# Example: Choose first record set and first numeric field
rec_set_id = first_record_set_id

# Identify numeric field by looking at the first record
numeric_field_id = None
df = dataframes[rec_set_id]
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Try grouping by another field (choose second column)
    group_field_id = None
    if len(df.columns) > 1:
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric field found in the selected record set.")

## 5. Visualization
Visualize numeric field distribution and relationship with a group field. Field references by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the normalized numeric field
if numeric_field_id and f"{numeric_field_id}_normalized" in filtered_df:
    plt.figure(figsize=(8,6))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
    plt.title(f"Distribution of normalized '{numeric_field_id}' (by @id)")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.show()

# Plot mean values grouped by group_field_id
if numeric_field_id and group_field_id:
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    plt.figure(figsize=(10,6))
    group_means.plot(kind='bar')
    plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}' (by @id)")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook guided users through loading, overview, and exploratory analysis of the FAIR^2 dataset using `mlcroissant`. All entities, including record sets, fields, and group keys, were referenced by their `@id`.

- Data loading and metadata review provided insight into dataset organization and provenance.
- Record sets and their fields were listed by their unique identifiers.
- Data from the record sets were loaded and processed, including filtering and normalization of numeric fields by column `@id`.
- Visualizations highlighted value distributions and relationships between grouped entities.

Further analysis can be performed by iterating over additional record sets, or connecting field-level metadata and annotations for deeper scientific or modeling objectives.